# AHP-MAUT Framework: Interactive Walkthrough

This notebook mirrors `run_all.py`, broken into explorable steps.

**Paper:** Silva et al. (2026). *Integrated AHP-MAUT Framework for Multicriteria Performance Evaluation of Soybean-Wheat Intercropping in Southern Brazil.* Submitted to *International Transactions in Operational Research*.

---

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.ahp import evaluate_matrix, load_pairwise_matrix
from src.maut import (apply_exponential_to_matrix, evaluate_scenarios,
                      global_utility)
from src.sensitivity import (monte_carlo_simulation, oat_sensitivity,
                             summarize_monte_carlo)

DATA_DIR = ROOT / 'data'
print(f'Data directory: {DATA_DIR}')
print(f'Files: {sorted(p.name for p in DATA_DIR.glob("*.csv"))}')

## 2. AHP weights and consistency check

Load the aggregated pairwise comparison matrix for the four main criteria and check internal consistency.

In [ ]:
main_matrix = load_pairwise_matrix(DATA_DIR / 'pairwise_main_criteria.csv')
print('Pairwise comparison matrix (main criteria):')
print(pd.DataFrame(main_matrix,
                   index=['E', 'A', 'En', 'L'],
                   columns=['E', 'A', 'En', 'L']).round(3))

result = evaluate_matrix(main_matrix)
print(f"\nlambda_max = {result['lambda_max']:.4f}")
print(f'CI         = {result["CI"]:.4f}')
print(f'CR         = {result["CR"]:.4f}  ({"OK" if result["consistent"] else "INCONSISTENT"})')
print(f'weights    = {result["weights"].round(4)}')

## 3. Load utility values and run baseline MAUT

Each row is a sub-criterion; columns are the three scenarios.

In [ ]:
weights_df = pd.read_csv(DATA_DIR / 'ahp_weights.csv')
utilities_df = pd.read_csv(DATA_DIR / 'utility_values.csv')

final_weights = weights_df['final_weight'].values
utility_matrix = utilities_df[['scenario_A', 'scenario_B', 'scenario_C']].values

baseline = evaluate_scenarios(final_weights, utility_matrix,
                              scenario_labels=['A', 'B', 'C'])
baseline['published'] = [0.8921, 0.5914, 0.2485]
baseline

## 4. Exponential utility (risk aversion)

Re-run MAUT with risk-averse utility functions to verify the ranking is robust to functional form choice.

In [ ]:
results_by_rho = {}
for rho in [0.5, 1.0, 2.0]:
    u_exp = apply_exponential_to_matrix(utility_matrix, rho=rho)
    sc = evaluate_scenarios(final_weights, u_exp, scenario_labels=['A', 'B', 'C'])
    results_by_rho[rho] = sc.set_index('scenario')['global_utility']

comparison = pd.DataFrame(results_by_rho)
comparison.columns = [f'rho={r}' for r in comparison.columns]
comparison.insert(0, 'linear', baseline.set_index('scenario')['global_utility'])
comparison.round(4)

## 5. Monte Carlo sensitivity

10,000 iterations, with uniform +/-20% perturbation on each main criterion weight, renormalized to sum to 1.

In [ ]:
main_weights = np.array([0.40, 0.25, 0.20, 0.15])
local_weights = [
    weights_df[weights_df.criterion == c]['local_weight'].values
    for c in ['Economic', 'Agronomic', 'Environmental', 'Logistical']
]

mc = monte_carlo_simulation(main_weights, local_weights, utility_matrix,
                            n_iterations=10000, perturbation_range=0.20,
                            random_seed=42)

summary = summarize_monte_carlo(mc)
print(summary.round(4).to_string(index=False))
print(f"\nRanking A > B > C preserved: {100*mc['ranking_preserved'].mean():.2f}%")

## 6. Visualize the Monte Carlo distributions

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for col, color in zip(['U_A', 'U_B', 'U_C'], ['#2C7BB6', '#FDAE61', '#D7191C']):
    ax.hist(mc[col], bins=50, alpha=0.6, label=col, color=color, edgecolor='black')
ax.set_xlabel('Global utility U(x)')
ax.set_ylabel('Frequency')
ax.set_title('Monte Carlo distribution of global utilities (10,000 iterations)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Reproduce the full paper figures

Or just run `python run_all.py` from the project root to regenerate everything in `results/`.